## Library Imports and Configs 

In [2]:
import numpy as np 
import pandas as pd 
pd.set_option('display.max_columns', None)
import os
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_PATH = "playground-series-s6e5"

train_df = pd.read_csv(os.path.join(ROOT_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(ROOT_PATH, "test.csv"))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

train_df.shape, test_df.shape # (70 % train, 30 % test)

((439140, 16), (188165, 15))

In [3]:
train_df.sample(4)

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
321243,321243,D027,MEDIUM,French Grand Prix,2022,0,1,1,1.0,3,106.959,29.977,-1.207,0.013889,-2.0,0.0
76757,76757,D018,HARD,Hungarian Grand Prix,2025,0,57,3,18.0,4,75.832,-0.081,-18.056,0.730769,0.0,0.0
183041,183041,FON,HARD,Monaco Grand Prix,2024,1,1,1,1.0,4,2497.268,24.080,2370.062,0.012821,-2.0,0.0
398004,398004,VIS,MEDIUM,Abu Dhabi Grand Prix,2025,0,43,3,7.0,6,88.167,-0.820,-3.707,0.551282,-1.0,0.0


In [4]:
# import itertools
# from scipy.stats import chi2_contingency
#
orig_df = train_df.copy()
# categorical_cols = ['Driver', 'Compound', 'Race', 'Year', 'Stint', 'TyreLife_bin', 'Position_bin']
#
# for col1, col2 in itertools.combinations(categorical_cols, 2):
#     if col1 in orig_df.columns and col2 in orig_df.columns:
#         orig_df['combined'] = orig_df[col1].astype(str) + ' + ' + orig_df[col2].astype(str)
#
#         tab = pd.crosstab(orig_df['combined'], orig_df['PitNextLap'])
#
#         chi2, p, dof, ex = chi2_contingency(tab)
#         print(f"{col1:10} + {col2:10} | chi2 = {chi2:10.2f} | p = {p:.6f} {'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else ''} | категорий={tab.shape[0]}")
#
# if 'combined' in orig_df.columns:
#     orig_df.drop('combined', axis=1, inplace=True)

import itertools
from scipy.stats import chi2_contingency

categorical_cols = [
    'Driver', 'Compound', 'Race',
    'Year', 'Stint', 'TyreLife_bin',
    'Position_bin'
]

for col1, col2 in itertools.combinations(categorical_cols, 2):

    if col1 in orig_df.columns and col2 in orig_df.columns:

        combined = (
            orig_df[col1].astype(str)
            + ' + ' +
            orig_df[col2].astype(str)
        )

        tab = pd.crosstab(combined, orig_df['PitNextLap'])

        chi2, p, dof, ex = chi2_contingency(tab)

        print(
            f"{col1:12} + {col2:12} | "
            f"chi2={chi2:10.2f} | "
            f"p={p:.6f}"
        )

Driver       + Compound     | chi2=  44602.62 | p=0.000000
Driver       + Race         | chi2=  36077.72 | p=0.000000
Driver       + Year         | chi2=  51118.45 | p=0.000000
Driver       + Stint        | chi2=  75345.00 | p=0.000000
Compound     + Race         | chi2=  62428.47 | p=0.000000
Compound     + Year         | chi2= 103770.53 | p=0.000000
Compound     + Stint        | chi2=  69207.94 | p=0.000000
Race         + Year         | chi2=  77672.74 | p=0.000000
Race         + Stint        | chi2=  87523.75 | p=0.000000
Year         + Stint        | chi2= 140402.25 | p=0.000000


In [5]:
orig_df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [6]:
target_counts = train_df['PitNextLap'].value_counts().to_dict()
print("Percentage count of 0.0 PitNextLap")
print(target_counts[0.0] / sum(list(target_counts.values())) * 100)
print("Percentage count of 1.0 PitNextLap")
print(target_counts[1.0] / sum(list(target_counts.values())) * 100)

Percentage count of 0.0 PitNextLap
80.101789862003
Percentage count of 1.0 PitNextLap
19.898210137996994


## Feature Engineering
- Domain Knowledge
- Correlation after feature engineering

In [7]:
# def engineer_race_features(df):
#     """
#     Applies feature engineering for race strategy prediction.
#     Handles high correlation via ratios and extracts stint-based metrics.
#     """
#     # Create a copy to avoid SettingWithCopyWarning
#     df = df.copy()

#     # 1. Handling your high correlation pair (RaceProgress / LapNumber)
#     # Adding a small epsilon to avoid division by zero
#     df['Progress_Per_Lap_engg'] = df['RaceProgress'] / (df['LapNumber'] + 1e-5)

#     # 2. Tyre & Degradation Ratios
#     # Captures the intensity of degradation relative to the distance traveled
#     df['Deg_Per_Lap_engg'] = df['Cumulative_Degradation'] / (df['LapNumber'] + 1e-5)
#     df['Deg_Per_TyreLife_engg'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1e-5)

#     # 4. Pace Sensitivity (The "Cliff" Detector)
#     # How much is the lap time changing relative to tyre age?
#     df['Pace_Tyre_Sensitivity_engg'] = df['LapTime_Delta'] / (df['TyreLife'] + 1e-5)

#     # 5. Strategic Flags
#     # Identify if a driver is losing positions (potential pressure to pit)
#     df['Losing_Ground_engg'] = (df['Position_Change'] < 0).astype(int)
    
#     # Identify 'Fresh' vs 'Old' tyres based on stint start
#     df['Is_Late_Stint_engg'] = (df['TyreLife'] > 20).astype(int) 

#     # 6. Interaction Terms
#     # Multiplying LapTime_Delta by Cumulative_Degradation to highlight 
#     # laps where both pace drops and wear is high
#     df['Wear_Pace_Impact_engg'] = df['LapTime_Delta'] * df['Cumulative_Degradation']

#     df['Year_Stint_engg'] = (
#     df['Year'].astype(str)
#     + "_" +
#     df['Stint'].astype(str)
#     )

#     df['Compound_Year_engg'] = (
#     df['Compound'].astype(str)
#     + "_" +
#     df['Year'].astype(str)
#     )

#     df['Race_Stint_engg'] = (
#     df['Race'].astype(str)
#     + "_" +
#     df['Stint'].astype(str)
#     )

#     return df

def engineer_race_features(df, activate=True):

    if not activate:
        return df

    df = df.copy()

    eps = 1e-5

    # =========================================================
    # BASIC PROGRESS FEATURES
    # =========================================================

    df['Progress_Per_Lap_engg'] = (
        df['RaceProgress'] / (df['LapNumber'] + eps)
    )

    df['Remaining_RaceProgress_engg'] = (
        1 - df['RaceProgress']
    )

    df['Remaining_Laps_Ratio_engg'] = (
        (1 - df['RaceProgress']) /
        (df['LapNumber'] + eps)
    )

    # =========================================================
    # DEGRADATION FEATURES
    # =========================================================

    df['Deg_Per_Lap_engg'] = (
        df['Cumulative_Degradation'] /
        (df['LapNumber'] + eps)
    )

    df['Deg_Per_TyreLife_engg'] = (
        df['Cumulative_Degradation'] /
        (df['TyreLife'] + eps)
    )

    df['Deg_x_TyreLife_engg'] = (
        df['Cumulative_Degradation'] *
        df['TyreLife']
    )

    df['Deg_x_Progress_engg'] = (
        df['Cumulative_Degradation'] *
        df['RaceProgress']
    )

    df['Deg_Acceleration_engg'] = (
        df['Cumulative_Degradation'] /
        (df['RaceProgress'] + eps)
    )

    # =========================================================
    # PACE FEATURES
    # =========================================================

    df['Pace_Tyre_Sensitivity_engg'] = (
        df['LapTime_Delta'] /
        (df['TyreLife'] + eps)
    )

    df['Pace_Per_Position_engg'] = (
        df['LapTime_Delta'] /
        (df['Position'] + eps)
    )

    df['LapTime_x_TyreLife_engg'] = (
        df['LapTime_Delta'] *
        df['TyreLife']
    )

    df['LapTime_x_Deg_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['LapTime_x_Progress_engg'] = (
        df['LapTime_Delta'] *
        df['RaceProgress']
    )

    df['Pace_Drop_Flag_engg'] = (
        df['LapTime_Delta'] > 0
    ).astype(int)

    df['Extreme_Pace_Drop_engg'] = (
        df['LapTime_Delta'] > df['LapTime_Delta'].quantile(0.90)
    ).astype(int)

    # =========================================================
    # TYRE FEATURES
    # =========================================================

    df['TyreLife_Per_Lap_engg'] = (
        df['TyreLife'] /
        (df['LapNumber'] + eps)
    )

    df['TyreLife_Per_Progress_engg'] = (
        df['TyreLife'] /
        (df['RaceProgress'] + eps)
    )

    df['TyreLife_x_Progress_engg'] = (
        df['TyreLife'] *
        df['RaceProgress']
    )

    df['Fresh_Tyre_Flag_engg'] = (
        df['TyreLife'] <= 5
    ).astype(int)

    df['Medium_Tyre_Flag_engg'] = (
        (df['TyreLife'] > 5) &
        (df['TyreLife'] <= 20)
    ).astype(int)

    df['Old_Tyre_Flag_engg'] = (
        df['TyreLife'] > 20
    ).astype(int)

    # =========================================================
    # POSITION FEATURES
    # =========================================================

    df['Losing_Ground_engg'] = (
        df['Position_Change'] < 0
    ).astype(int)

    df['Gaining_Ground_engg'] = (
        df['Position_Change'] > 0
    ).astype(int)

    df['Position_x_Progress_engg'] = (
        df['Position'] *
        df['RaceProgress']
    )

    df['Position_x_TyreLife_engg'] = (
        df['Position'] *
        df['TyreLife']
    )

    df['Position_Change_Intensity_engg'] = (
        df['Position_Change'] /
        (df['LapNumber'] + eps)
    )

    df['Bad_Position_Flag_engg'] = (
        df['Position'] > 10
    ).astype(int)

    df['Podium_Position_Flag_engg'] = (
        df['Position'] <= 3
    ).astype(int)

    # =========================================================
    # PIT WINDOW FEATURES
    # =========================================================

    df['Potential_Pit_Window_engg'] = (
        (df['TyreLife'] > 15) &
        (df['RaceProgress'] > 0.25)
    ).astype(int)

    df['Late_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] > 0.70)
    ).astype(int)

    df['Early_Race_Pit_Window_engg'] = (
        (df['RaceProgress'] < 0.25)
    ).astype(int)

    # =========================================================
    # INTERACTION FEATURES
    # =========================================================

    df['Wear_Pace_Impact_engg'] = (
        df['LapTime_Delta'] *
        df['Cumulative_Degradation']
    )

    df['Wear_Position_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position']
    )

    df['Wear_Position_Change_Impact_engg'] = (
        df['Cumulative_Degradation'] *
        df['Position_Change']
    )

    df['TyreLife_Position_Interaction_engg'] = (
        df['TyreLife'] *
        df['Position']
    )

    df['TyreLife_Lap_Interaction_engg'] = (
        df['TyreLife'] *
        df['LapNumber']
    )

    df['TyreLife_Stint_Interaction_engg'] = (
        df['TyreLife'] *
        df['Stint']
    )

    df['Lap_Position_Interaction_engg'] = (
        df['LapNumber'] *
        df['Position']
    )

    # =========================================================
    # STINT FEATURES
    # =========================================================

    df['Is_First_Stint_engg'] = (
        df['Stint'] == 1
    ).astype(int)

    df['Is_Second_Stint_engg'] = (
        df['Stint'] == 2
    ).astype(int)

    df['Is_ThirdPlus_Stint_engg'] = (
        df['Stint'] >= 3
    ).astype(int)

    df['Stint_x_Progress_engg'] = (
        df['Stint'] *
        df['RaceProgress']
    )

    df['Stint_x_TyreLife_engg'] = (
        df['Stint'] *
        df['TyreLife']
    )

    # =========================================================
    # CATEGORICAL COMBINATIONS
    # =========================================================

    df['Year_Stint_engg'] = (
        df['Year'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Year_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Race_Stint_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Driver_Compound_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Driver_Race_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Race'].astype(str)
    )

    df['Compound_Stint_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    df['Compound_Position_engg'] = (
        df['Compound'].astype(str)
        + "_"
        + df['Position'].astype(str)
    )

    df['Race_Compound_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Compound'].astype(str)
    )

    df['Race_Year_engg'] = (
        df['Race'].astype(str)
        + "_"
        + df['Year'].astype(str)
    )

    df['Driver_Stint_engg'] = (
        df['Driver'].astype(str)
        + "_"
        + df['Stint'].astype(str)
    )

    return df
 
# Apply to your dataframes
train_df_eng = engineer_race_features(train_df)
test_df_eng = engineer_race_features(test_df)

train_df_eng.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Year_Stint_engg,Compound_Year_engg,Race_Stint_engg,Driver_Compound_engg,Driver_Race_engg,Compound_Stint_engg,Compound_Position_engg,Race_Compound_engg,Race_Year_engg,Driver_Stint_engg
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,0.014286,0.285714,0.005714,0.420380,0.538949,819.741,15.013571,29.426188,-0.193949,-0.945499,-294.996,-158.987716,-5.402857,0,0,0.780000,54.599236,27.857143,0,0,1,0,1,5.714286,312.0,0.100000,0,0,1,1,0,-158.987716,168.152,105.095,312.0,1950.0,78.0,400,0,1,0,1.428571,78.0,2022_2,HARD_2022,Canadian Grand Prix_2,D109_HARD,D109_Canadian Grand Prix,HARD_2,HARD_8,Canadian Grand Prix_HARD,Canadian Grand Prix_2022,D109_2
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0,0.012821,0.653846,0.024217,-8.266923,-31.886669,-1562.449,-77.263962,-644.801595,-4.659565,-8.154230,-228.319,7280.342719,-11.290500,0,0,0.259259,20.221638,2.423077,0,1,0,1,0,1.384615,28.0,-0.111111,0,0,0,0,0,7280.342719,-892.828,669.621,28.0,189.0,14.0,108,0,1,0,0.692308,14.0,2025_2,HARD_2025,Dutch Grand Prix_2,D086_HARD,D086_Dutch Grand Prix,HARD_2,HARD_4,Dutch Grand Prix_HARD,Dutch Grand Prix_2025,D086_2
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,0.013889,0.180556,0.003060,-1.703881,-4.569498,-2211.638,-82.377931,-122.677961,-0.342727,-0.580000,-165.880,757.988660,-6.178611,0,0,0.372881,26.847130,18.027778,0,0,1,0,1,10.652778,286.0,0.050847,1,0,1,1,0,757.988660,-1306.877,-301.587,286.0,1298.0,66.0,767,0,0,1,2.458333,66.0,2022_3,HARD_2022,Austrian Grand Prix_3,ZON_HARD,ZON_Austrian Grand Prix,HARD_3,HARD_13,Austrian Grand Prix_HARD,Austrian Grand Prix_2022,ZON_3
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0.038461,0.923077,0.461536,-3.661982,-3.661982,-14.648,-0.563385,-95.199624,-3.661982,-1.046284,-14.648,53.640976,-0.563385,0,0,0.999995,25.996620,0.153846,1,0,0,0,0,0.538462,14.0,0.000000,0,0,0,0,1,53.640976,-51.268,-0.000,14.0,4.0,2.0,14,1,0,0,0.076923,2.0,2023_1,MEDIUM_2023,Pre-Season Testing_1,SPE_MEDIUM,SPE_Pre-Season Testing,MEDIUM_1,MEDIUM_7,Pre-Season Testing_MEDIUM,Pre-Season Testing_2023,SPE_1
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,0.013889,0.638889,0.024573,-0.543807,-2.356496,-84.834,-5.105750,-39.153070,1.494164,4.482478,53.790,-126.756135,3.237361,1,0,0.230769,16.614925,2.166667,0,1,0,0,1,0.722222,12.0,0.115385,0,1,0,0,0,-126.756135,-28.278,-42.417,12.0,156.0,18.0,52,0,0,1,1.083333,18.0,2022_3,HARD_2022,Azerbaijan Grand Prix_3,D019_HARD,D019_Azerbaijan Grand Prix,HARD_3,HARD_2,Azerbaijan Grand Prix_HARD,Azerbaijan Grand Prix_2022,D019_3


In [8]:
cat_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(exclude=np.number).columns
num_cols = test_df_eng.drop(['id'], axis='columns').select_dtypes(include=np.number).columns
target_col = ['PitNextLap']

## Data Processing
- Encoding - (categorical data) (Label Encoding and OneHotEncoding)
- Normalization - (numerical data)

#### Encoding - Categorical data - Label Encoding and OneHotEncoding

In [9]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoder = LabelEncoder()

train_df_eng["Driver_en"] = label_encoder.fit_transform(train_df_eng['Driver'])
test_df_eng["Driver_en"] = label_encoder.transform(test_df_eng['Driver'])

train_df_eng["Race_en"] = label_encoder.fit_transform(train_df_eng['Race'])
test_df_eng["Race_en"] = label_encoder.transform(test_df_eng['Race'])

    
train_df_eng_enc = train_df_eng.drop(["Driver", "Race"], axis='columns')
test_df_eng_enc = test_df_eng.drop(["Driver", "Race"], axis='columns')

In [10]:
one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_data = one_hot_encoder.fit_transform(train_df_eng_enc[['Compound']])
encoded_df = pd.DataFrame(
    encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=train_df_eng_enc.index
)
train_df_eng_enc_v1 = pd.concat([train_df_eng_enc, encoded_df], axis=1).drop('Compound', axis=1)


test_encoded_data = one_hot_encoder.transform(test_df_eng_enc[['Compound']])
test_encoded_df = pd.DataFrame(
    test_encoded_data, 
    columns=one_hot_encoder.get_feature_names_out(['Compound']),
    index=test_df_eng_enc.index
)
test_df_eng_enc_v1 = pd.concat([test_df_eng_enc, test_encoded_df], axis=1).drop('Compound', axis=1)

train_df_eng_enc_v1.filter(like='Compound').head()

,Compound_Year_engg,Driver_Compound_engg,Compound_Stint_engg,Compound_Position_engg,Race_Compound_engg,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,HARD_2022,D109_HARD,HARD_2,HARD_8,Canadian Grand Prix_HARD,1.0,0.0,0.0,0.0,0.0
1,HARD_2025,D086_HARD,HARD_2,HARD_4,Dutch Grand Prix_HARD,1.0,0.0,0.0,0.0,0.0
2,HARD_2022,ZON_HARD,HARD_3,HARD_13,Austrian Grand Prix_HARD,1.0,0.0,0.0,0.0,0.0
3,MEDIUM_2023,SPE_MEDIUM,MEDIUM_1,MEDIUM_7,Pre-Season Testing_MEDIUM,0.0,0.0,1.0,0.0,0.0
4,HARD_2022,D019_HARD,HARD_3,HARD_2,Azerbaijan Grand Prix_HARD,1.0,0.0,0.0,0.0,0.0


#### Normalization (MinMax / StandardScaler)

In [11]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

class DataNormalizer:
    def __init__(self, method='standard'):
        """
        Initializes the normalizer with the chosen scaling strategy.
        method: 'minmax' or 'standard'
        """
        self.method = method
        if method == 'minmax':
            self.scaler = MinMaxScaler()
        elif method == 'standard':
            self.scaler = StandardScaler()
        else:
            raise ValueError("Method must be either 'minmax' or 'standard'")

    def fit(self, dataset, num_cols):
        """
        Learns the scaling parameters (mean/std or min/max) from the training set.
        """
        if not all(col in dataset.columns for col in num_cols):
            missing = [c for c in num_cols if c not in dataset.columns]
            raise ValueError(f"Columns missing from dataset: {missing}")
            
        self.scaler.fit(dataset[num_cols])
        print(f"Successfully fitted {self.method} scaler on: {num_cols}")

    def transform(self, dataset, num_cols):
        """
        Applies the learned parameters to scale the dataset.
        """
        df = dataset.copy()
        df[num_cols] = self.scaler.transform(df[num_cols])
        return df

    def fit_transform(self, dataset, num_cols):
        """
        Fits to the data then transforms it. Useful for the initial training set.
        """
        self.fit(dataset, num_cols)
        return self.transform(dataset, num_cols)


normalizer = DataNormalizer(method='standard')

train_df_scaled = normalizer.fit_transform(train_df_eng_enc_v1, num_cols=num_cols)
test_df_scaled = normalizer.transform(test_df_eng_enc_v1, num_cols=num_cols)

train_df_scaled.head()

Successfully fitted standard scaler on: Index(['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'Progress_Per_Lap_engg',
       'Remaining_RaceProgress_engg', 'Remaining_Laps_Ratio_engg',
       'Deg_Per_Lap_engg', 'Deg_Per_TyreLife_engg', 'Deg_x_TyreLife_engg',
       'Deg_x_Progress_engg', 'Deg_Acceleration_engg',
       'Pace_Tyre_Sensitivity_engg', 'Pace_Per_Position_engg',
       'LapTime_x_TyreLife_engg', 'LapTime_x_Deg_engg',
       'LapTime_x_Progress_engg', 'Pace_Drop_Flag_engg',
       'Extreme_Pace_Drop_engg', 'TyreLife_Per_Lap_engg',
       'TyreLife_Per_Progress_engg', 'TyreLife_x_Progress_engg',
       'Fresh_Tyre_Flag_engg', 'Medium_Tyre_Flag_engg', 'Old_Tyre_Flag_engg',
       'Losing_Ground_engg', 'Gaining_Ground_engg', 'Position_x_Progress_engg',
       'Position_x_TyreLife_engg', 'Position_Change_Intensity_engg',
       'Bad_Position_Flag_engg', 

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Year_Stint_engg,Compound_Year_engg,Race_Stint_engg,Driver_Compound_engg,Driver_Race_engg,Compound_Stint_engg,Compound_Position_engg,Race_Compound_engg,Race_Year_engg,Driver_Stint_engg,Driver_en,Race_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,0,-1.486487,-0.396946,1.585901,0.221941,2.534531,-0.308849,-0.630046,-0.086333,0.853455,1.487008,1.222548,1.0,-0.091033,-1.487008,-0.509986,0.151346,0.220112,1.196900,1.220405,0.142041,0.024664,-0.023204,-1.253837,-0.019963,-0.979879,-0.660523,-0.333333,-0.019388,-0.019540,2.812363,-0.509979,-1.130327,1.816110,-0.688951,1.320038,0.745101,1.515992,0.098721,-0.899887,-0.433810,1.403299,2.817278,-0.925252,-0.019963,0.747463,0.380384,1.515992,2.875356,2.040728,0.802183,-0.985163,1.545995,-0.519458,0.741509,2.040728,2022_2,HARD_2022,Canadian Grand Prix_2,D109_HARD,D109_Canadian Grand Prix,HARD_2,HARD_8,Canadian Grand Prix_HARD,Canadian Grand Prix_2022,D109_2,134,7,1.0,0.0,0.0,0.0,0.0
1,1,1.440545,2.519236,0.229628,0.221941,-0.730333,-1.066602,-0.801797,-0.656423,-3.605949,0.033534,-0.774077,0.0,-0.401788,-0.033534,-0.416683,-0.384754,-1.788677,-1.095286,-3.176571,-0.405659,-0.088712,-0.585639,-0.928667,0.365547,-2.250847,-0.660523,-0.333333,-1.234165,-1.149458,-0.509855,-0.509979,0.884700,-0.550627,1.451482,-0.757554,-0.569919,-0.850941,-0.099932,-0.899887,-0.433810,-0.712606,-0.354953,-0.925252,0.365547,-1.124840,2.373826,-0.850941,-0.466697,-0.510165,-0.518896,-0.985163,1.545995,-0.519458,-0.093803,-0.510165,2025_2,HARD_2025,Dutch Grand Prix_2,D086_HARD,D086_Dutch Grand Prix,HARD_2,HARD_4,Dutch Grand Prix_HARD,Dutch Grand Prix_2025,D086_2,111,9,1.0,0.0,0.0,0.0,0.0
2,2,-1.486487,-0.396946,2.116616,1.274359,0.800072,0.638343,-1.011682,-0.085787,-1.365930,1.902200,0.723392,1.0,-0.175196,-1.902200,-0.523369,0.020257,-0.096360,-1.719948,-3.420249,0.018482,0.020887,0.005312,-0.624164,0.027555,-1.147342,-0.660523,-0.333333,-0.969109,-0.931692,1.528443,-0.509979,-1.130327,1.816110,-0.688951,1.320038,2.245034,1.299301,0.052469,1.111250,-0.433810,1.403299,2.817278,-0.925252,0.027555,-1.855510,-1.055683,1.299301,1.637980,1.562435,2.462581,-0.985163,-0.646833,1.925083,1.909803,1.562435,2022_3,HARD_2022,Austrian Grand Prix_3,ZON_HARD,ZON_Austrian Grand Prix,HARD_3,HARD_13,Austrian Grand Prix_HARD,Austrian Grand Prix_2022,ZON_3,886,2,1.0,0.0,0.0,0.0,0.0
3,3,-0.510810,-0.396946,-1.244581,-0.830476,-1.240468,-0.498287,0.172574,-0.080872,0.335931,-1.029455,-0.025343,0.0,5.036382,1.029455,1.788622,-0.100579,-0.040139,0.394036,0.478171,0.040803,-0.063385,-0.031068,0.113365,-0.008944,0.064819,-0.660523,-0.333333,0.493814,-0.959647,-0.806264,1.960864,-1.130327,-0.550627,-0.688951,-0.757554,-0.826915,-0.967621,0.004622,-0.899887,-0.433810,-0.712606,-0.354953,1.080787,-0.008944,0.360254,0.009275,-0.967621,-0.817793,-0.9

In [12]:
train_df_scaled.shape, test_df_scaled.shape

((439140, 73), (188165, 72))

## Choosing Best Model 

In [13]:
def submission(model, test_data, file_name:str):
    pred_data = test_data
    if 'id' in test_data.columns:
        pred_data = test_data.drop(['id'], axis='columns')
        print("!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!")
        
    predictions = model.predict_proba(pred_data)[:, 1]
    sub_df['PitNextLap'] = predictions
    sub_df[['id', 'PitNextLap']].to_csv(file_name, index=False)
    print(f"Submissions saved to {file_name} path!!!!!!!!!!!!!!")

In [14]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Target Encoding - OOF

In [15]:
encoded_cat_cols = [col for col in train_df_scaled.columns if '_en' in col.lower() and '_engg' not in col.lower()]
encoded_cat_cols 

['Driver_en', 'Race_en']

In [16]:
from sklearn.model_selection import KFold

train_df_scaled_v1 = train_df_scaled.copy()
test_df_scaled_v1 = test_df_scaled.copy()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in encoded_cat_cols:
    train_df_scaled_v1[f'{col}_mean'] = 0.0
    train_df_scaled_v1[f'{col}_std'] = 0.0

for train_idx, val_idx in kf.split(train_df_scaled_v1):
    train_fold = train_df_scaled_v1.iloc[train_idx]
    val_fold = train_df_scaled_v1.iloc[val_idx]

    for col in encoded_cat_cols:
        stats = train_fold.groupby(col)[target_col[0]].agg(['mean', 'std'])
        val_fold = val_fold.merge(stats, on=col, how='left')

        train_df_scaled_v1.loc[val_idx, f'{col}_mean'] = val_fold['mean'].values
        train_df_scaled_v1.loc[val_idx, f'{col}_std'] = val_fold['std'].values

        val_fold.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    train_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = train_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

for col in encoded_cat_cols:
    stats = train_df_scaled_v1.groupby(col)[target_col[0]].agg(['mean', 'std'])
    test_df_scaled_v1 = test_df_scaled_v1.merge(stats, on=col, how='left')

    test_df_scaled_v1[f'{col}_mean'] = test_df_scaled_v1['mean']
    test_df_scaled_v1[f'{col}_std'] = test_df_scaled_v1['std']

    test_df_scaled_v1.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    test_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = test_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)


X, y = train_df_scaled_v1.drop(['id', 'PitNextLap'], axis='columns'), train_df_scaled_v1['PitNextLap'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

In [18]:
X_train.head()

,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,Progress_Per_Lap_engg,Remaining_RaceProgress_engg,Remaining_Laps_Ratio_engg,Deg_Per_Lap_engg,Deg_Per_TyreLife_engg,Deg_x_TyreLife_engg,Deg_x_Progress_engg,Deg_Acceleration_engg,Pace_Tyre_Sensitivity_engg,Pace_Per_Position_engg,LapTime_x_TyreLife_engg,LapTime_x_Deg_engg,LapTime_x_Progress_engg,Pace_Drop_Flag_engg,Extreme_Pace_Drop_engg,TyreLife_Per_Lap_engg,TyreLife_Per_Progress_engg,TyreLife_x_Progress_engg,Fresh_Tyre_Flag_engg,Medium_Tyre_Flag_engg,Old_Tyre_Flag_engg,Losing_Ground_engg,Gaining_Ground_engg,Position_x_Progress_engg,Position_x_TyreLife_engg,Position_Change_Intensity_engg,Bad_Position_Flag_engg,Podium_Position_Flag_engg,Potential_Pit_Window_engg,Late_Race_Pit_Window_engg,Early_Race_Pit_Window_engg,Wear_Pace_Impact_engg,Wear_Position_Impact_engg,Wear_Position_Change_Impact_engg,TyreLife_Position_Interaction_engg,TyreLife_Lap_Interaction_engg,TyreLife_Stint_Interaction_engg,Lap_Position_Interaction_engg,Is_First_Stint_engg,Is_Second_Stint_engg,Is_ThirdPlus_Stint_engg,Stint_x_Progress_engg,Stint_x_TyreLife_engg,Year_Stint_engg,Compound_Year_engg,Race_Stint_engg,Driver_Compound_engg,Driver_Race_engg,Compound_Stint_engg,Compound_Position_engg,Race_Compound_engg,Race_Year_engg,Driver_Stint_engg,Driver_en,Race_en,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET,Driver_en_mean,Driver_en_std,Race_en_mean,Race_en_std
196348,0.464868,2.519236,-1.126644,-0.830476,-1.036414,-0.119410,0.381346,-0.131980,-0.040777,-1.128062,0.723392,-0.366481,1.128062,0.656399,-0.305876,-0.246233,0.300535,0.435820,-0.318925,-0.031154,-0.032398,-0.001884,0.002139,0.079119,-0.660523,-0.333333,0.493819,0.716239,-0.799218,1.960864,-1.130327,-0.550627,-0.688951,1.320038,-0.848458,-0.784267,0.710360,-0.899887,-0.43381,-0.712606,-0.354953,1.080787,0.002139,0.006739,-0.286867,-0.784267,-0.795019,-0.908743,-0.844642,1.015061,-0.646833,-0.519458,-0.820309,-0.908743,2024_1,MEDIUM_2024,Singapore Grand Prix_1,GLO_MEDIUM,GLO_Singapore Grand Prix,MEDIUM_1,MEDIUM_9,Singapore Grand Prix_MEDIUM,Singapore Grand Prix_2024,GLO_1,799,22,0.0,0.0,1.0,0.0,0.0,0.237542,0.425754,0.143037,0.350122
267152,1.440545,-0.396946,-0.006245,0.221941,-0.118171,1.206658,-0.377425,0.110114,-3.412893,-0.138303,-0.274921,-0.330233,0.138303,-0.385903,-0.445109,-0.826569,-2.251674,-2.561216,-0.452606,0.031676,0.055777,0.252573,-0.023503,0.256274,1.513952,-0.333333,-0.520430,-0.402257,-0.312470,-0.509979,0.884700,-0.550627,1.451482,-0.757554,0.480200,0.649228,-0.036290,1.111250,-0.43381,-0.712606,-0.354953,-0.925252,-0.023503,-5.553011,0.760123,0.649228,-0.257938,-0.031873,0.657407,-0.985163,1.545995,-0.519458,-0.192557,-0.031873,2025_2,HARD_2025,Italian Grand Prix_2,ERI_HARD,ERI_Italian Grand Prix,HARD_2,HARD_16,Italian Grand Prix_HARD,Italian Grand Prix_2025,ERI_2,790,13,1.0,0.0,0.0,0.0,0.0,0.215440,0.411312,0.131974,0.338473
108695,0.464868,2.519236,0.288597,1.274359,-1.138441,1.585534,0.149006,-0.025508,-0.319415,0.223888,-1.273234,-0.133708,-0.223888,-0.429727,0.030160,-0.705673,0.283384,-0.307051,0.029123,-0.011804,0.029364,0.113243,-0.000771,-0.229943,-0.660523,-0.333333,-1.589020,-1.564076,-0.671822,1.960864,-1.130327,-0.550627,1.451482,-0.757554,1.165546,-0.634250,-0.163411,1.111250,-0.43381,-0.712606,-0.354953,-0.925252,-0.000771,-0.921975,0.772274,-0.634250,-0.665968,-0.709454,1.272705,-0.985163,-0.646833,1.925083,0.463013,-0.709454,2024_3,MEDIUM_2024,Miami Grand Prix_3,D025_MEDIUM,D025_Miami Grand Prix,MEDIUM_3,MEDIUM_18,Miami Grand Prix_MEDIUM,Miami Grand Prix_2024,D025_3,50,17,0.0,0.0,1.0,0.0,0.0,0.187612,0.390578,0.102495,0.303308
92542,1.440545,2.519236,-0.301087,-0.830476,0.391964,-0.308849,0.197507,-0.005756,2.576972,-0.410200,-1.023655,-0.366476,0.410200,-0.324138,0.521074,0.583930,2.407024,1.790550,0.519170,0.023914,0.011330,-0.168349,-0.035784,-0.016576,-0.660523,-0.333333,0.493824,0.716618,-0.276734,-0.509979,0.8

In [17]:
count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

xgb_model = XGBClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    max_depth         = 7,
    min_child_weight  = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1,
    eval_metric       = 'auc',
    verbosity         = 0,
    scale_pos_weight = scale_weight
)

xgb_model.fit(X_train.values, y_train.values)

# --- XGBoost Evaluation ---
y_prob_test_xgb = xgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_xgb = xgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - XGBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")

ValueError: could not convert string to float: 'MEDIUM_2024'

In [ ]:
# xgb_model_final_v1 = XGBClassifier(
#     n_estimators      = 2000,
#     learning_rate     = 0.02,
#     max_depth         = 7,
#     min_child_weight  = 20,
#     subsample         = 0.8,
#     colsample_bytree  = 0.8,
#     reg_alpha         = 0.1,
#     reg_lambda        = 0.1,
#     random_state      = 42,
#     n_jobs            = -1,
#     eval_metric       = 'auc',
#     verbosity         = 0,
#     scale_pos_weight = scale_weight
# )
#
# xgb_model_final_v1.fit(X.values, y.values)
# submission(xgb_model_final_v1, test_df_scaled_v1, file_name="First_XGB_solution.csv")

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1
)


lgb_model.fit(X_train.values, y_train.values)

# --- LightGBM Evaluation ---
y_prob_test_lgb = lgb_model.predict_proba(X_test.values)[:, 1]
y_prob_train_lgb = lgb_model.predict_proba(X_train.values)[:, 1]

print("ROC AUC Score - LightGBM")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")

[LightGBM] [Info] Number of positive: 61167, number of negative: 246231
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032665 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3581
[LightGBM] [Info] Number of data points in the train set: 307398, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392662
[LightGBM] [Info] Start training from score -1.392662


C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\admin\Desktop\kaggle_competition\comp_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC Score - LightGBM
Testing:  0.9504
Training: 0.9858


In [ ]:
lgb_model_final_v1 = LGBMClassifier(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1
)
lgb_model_final_v1.fit(X.values, y.values)
submission(lgb_model_final_v1, test_df_scaled_v1, file_name="First_LGB_solution_11_05_2026.csv")

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055987 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3580
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to First_LGB_solution_11_05_2026.csv path!!!!!!!!!!!!!!


In [ ]:
model_names = ['xgb', 'lgb']
model_scores = [roc_auc_score(y_test, y_prob_test_xgb), roc_auc_score(y_test, y_prob_test_lgb)]
weights = [score / sum(model_scores) for score in model_scores]
model_weights = dict(zip(model_names, weights))

lgb_predictions = pd.read_csv("First_LGB_solution_11_05_2026.csv")
xgb_predictions = pd.read_csv("First_XGB_solution.csv")

final_prediction = model_weights['xgb'] * xgb_predictions['PitNextLap'] + model_weights['lgb'] * lgb_predictions['PitNextLap']
final_predictions_df = pd.DataFrame({'id': test_df_scaled_v1['id'], 'PitNextLap':final_prediction})
final_predictions_df.head()

,id,PitNextLap
0,439140,0.009414
1,439141,0.009038
2,439142,0.008191
3,439143,0.270456
4,439144,0.917215


In [ ]:
final_predictions_df.to_csv("Blending_XGB_LGB_11_05_2026.csv", index=False)

In [ ]:
cat_model_oot = CatBoostClassifier(
    iterations=20000,
    verbose=0,
    auto_class_weights='Balanced'
)

cat_model_oot.fit(X_train, y_train)

y_prob_test_cat = cat_model_oot.predict_proba(X_test)[:, 1]
y_prob_train_cat = cat_model_oot.predict_proba(X_train)[:, 1]

print("ROC AUC Score - CatBoost")
print(f"Testing:  {roc_auc_score(y_test, y_prob_test_cat):.4f}")
print(f"Training: {roc_auc_score(y_train, y_prob_train_cat):.4f}")

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=54]="2024_1": Cannot convert '2024_1' to float

In [ ]:
# cat_model_final_v1 = CatBoostClassifier(
#     iterations=20000,
#     eval_metric = "AUC",
#     verbose=0,
#     auto_class_weights='Balanced'
# )
#
# cat_model_final_v1.fit(X, y)
# submission(cat_model_final_v1, test_df_scaled_v1, file_name="CatBoost_Solution.csv")

!!!!!!!!!!!!!!!!!! Removed id column !!!!!!!!!!!!!!!!!!!
Submissions saved to CatBoost_Solution.csv path!!!!!!!!!!!!!!


In [ ]:
model_names = ['xgb', 'lgb', 'cat']
model_scores = [roc_auc_score(y_test, y_prob_test_xgb), roc_auc_score(y_test, y_prob_test_lgb), roc_auc_score(y_test, y_prob_test_cat)]
weights = [score / sum(model_scores) for score in model_scores]
model_weights = dict(zip(model_names, weights))

lgb_predictions = pd.read_csv("First_LGB_solution_11_05_2026.csv")
xgb_predictions = pd.read_csv("First_XGB_solution.csv")
cat_predictions = pd.read_csv("CatBoost_Solution.csv")

final_predictions_v1 = model_weights['xgb'] * xgb_predictions['PitNextLap'] + model_weights['lgb'] * lgb_predictions['PitNextLap'] + model_weights['cat'] * cat_predictions['PitNextLap']
final_predictions_v1_df = pd.DataFrame({'id': test_df_scaled_v1['id'], 'PitNextLap':final_predictions_v1})
final_predictions_v1_df.head()

,id,PitNextLap
0,439140,0.012803
1,439141,0.013318
2,439142,0.012014
3,439143,0.293793
4,439144,0.923788


In [ ]:
final_predictions_v1_df.to_csv("Blending_XGB_LGB_CAT_11_05_2026.csv", index=False)